# FraudShieldAI — Exploratory Data Analysis

**Dataset:** Credit Card Transactions Fraud Detection (Kaggle)  
**Rows:** ~1.3 million transactions  
**Goal:** Understand the data distribution, fraud patterns, feature relationships, and class imbalance before modelling.

## Outline
1. [Setup & Load](#1-setup)
2. [Dataset Overview](#2-overview)
3. [Class Imbalance](#3-imbalance)
4. [Transaction Amount Analysis](#4-amounts)
5. [Temporal Patterns](#5-temporal)
6. [Geographic Analysis](#6-geo)
7. [Merchant & Category Analysis](#7-merchant)
8. [Cardholder Demographics](#8-demographics)
9. [Feature Correlations](#9-correlations)
10. [Engineered Feature Preview](#10-features)
11. [Key Findings & Modelling Implications](#11-findings)

## 1 — Setup & Load <a id='1-setup'></a>

In [ ]:
import sys
from pathlib import Path

# Project root on path
ROOT = Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
FRAUD_COLOR  = '#e74c3c'
LEGIT_COLOR  = '#3498db'
PALETTE      = {'Fraud': FRAUD_COLOR, 'Legit': LEGIT_COLOR}

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

In [ ]:
train = pd.read_csv('../data/raw/fraudTrain.csv', parse_dates=['trans_date_trans_time'])
test  = pd.read_csv('../data/raw/fraudTest.csv',  parse_dates=['trans_date_trans_time'])
df    = pd.concat([train, test], ignore_index=True)

df['label'] = df['is_fraud'].map({0: 'Legit', 1: 'Fraud'})

print(f'Total rows : {len(df):,}')
print(f'Train rows : {len(train):,}  |  Test rows: {len(test):,}')
print(f'Columns    : {df.shape[1]}')

## 2 — Dataset Overview <a id='2-overview'></a>

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — dataset is complete.')

In [ ]:
df.describe(include='all').T

In [ ]:
print(f"Date range : {df['trans_date_trans_time'].min()} → {df['trans_date_trans_time'].max()}")
print(f"Unique cards    : {df['cc_num'].nunique():,}")
print(f"Unique merchants: {df['merchant'].nunique():,}")
print(f"Categories      : {df['category'].nunique()}")

## 3 — Class Imbalance <a id='3-imbalance'></a>

Fraud detection datasets are extremely imbalanced by nature.  
We quantify this before deciding on a sampling strategy.

In [ ]:
counts = df['is_fraud'].value_counts()
pct    = df['is_fraud'].value_counts(normalize=True) * 100

imbalance_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pct.round(3)})
imbalance_df.index = ['Legit (0)', 'Fraud (1)']
print(imbalance_df)
print(f"\nImbalance ratio  : {counts[0] / counts[1]:.0f}:1  (legit:fraud)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legit', 'Fraud'], counts.values,
            color=[LEGIT_COLOR, FRAUD_COLOR], edgecolor='white', linewidth=1.5)
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for i, v in enumerate(counts.values):
    axes[0].text(i, v * 1.01, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=['Legit', 'Fraud'],
            colors=[LEGIT_COLOR, FRAUD_COLOR],
            autopct='%1.2f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution')

plt.suptitle('Class Imbalance Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Finding:** The dataset is severely imbalanced — only ~0.57% of transactions are fraudulent (~427:1 ratio). Standard accuracy is meaningless here; we will use **AUC-PR** and **F1** as primary metrics, and apply **SMOTE** during training.

## 4 — Transaction Amount Analysis <a id='4-amounts'></a>

In [ ]:
print('Amount statistics by class:')
df.groupby('label')['amt'].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Log-scale histogram
clip = df['amt'].quantile(0.99)
for lbl, color in PALETTE.items():
    subset = df[df['label'] == lbl]['amt'].clip(upper=clip)
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=lbl, density=True)
axes[0].set_title('Amount Distribution (clipped at 99th pct)')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Log-transformed
for lbl, color in PALETTE.items():
    subset = np.log1p(df[df['label'] == lbl]['amt'])
    axes[1].hist(subset, bins=60, alpha=0.6, color=color, label=lbl, density=True)
axes[1].set_title('log(1 + Amount) Distribution')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].legend()

# Box plot
df_clip = df[df['amt'] < clip]
groups  = [df_clip[df_clip['label'] == lbl]['amt'] for lbl in ['Legit', 'Fraud']]
bp = axes[2].boxplot(groups, labels=['Legit', 'Fraud'],
                     patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], [LEGIT_COLOR, FRAUD_COLOR]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Amount Box Plot')
axes[2].set_ylabel('Amount ($)')

plt.suptitle('Transaction Amount: Fraud vs Legit', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Amount bins
bins = [0, 10, 50, 100, 250, 500, 1000, np.inf]
labels = ['<$10', '$10–50', '$50–100', '$100–250', '$250–500', '$500–1k', '>$1k']
df['amt_bin'] = pd.cut(df['amt'], bins=bins, labels=labels)

fraud_by_bin = df.groupby('amt_bin')['is_fraud'].agg(['mean', 'sum', 'count'])
fraud_by_bin.columns = ['fraud_rate', 'fraud_count', 'total']
print(fraud_by_bin.round(4))

> **Finding:** Fraudulent transactions tend to cluster in specific amount ranges (often mid-range). The log-transformed distribution shows fraud has a more uniform spread — fraudsters vary amounts deliberately to avoid detection.

## 5 — Temporal Patterns <a id='5-temporal'></a>

In [ ]:
df['hour']        = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.day_name()
df['month_year']  = df['trans_date_trans_time'].dt.to_period('M').astype(str)
df['date']        = df['trans_date_trans_time'].dt.date

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# --- Hour of day ---
hour_rate = df.groupby('hour')['is_fraud'].mean()
hour_cnt  = df.groupby(['hour', 'label']).size().unstack(fill_value=0)
hour_cnt.plot(kind='bar', ax=axes[0, 0], color=[LEGIT_COLOR, FRAUD_COLOR],
              width=0.8, edgecolor='none', legend=True)
axes[0, 0].set_title('Transaction Volume by Hour')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Count')
ax2 = axes[0, 0].twinx()
ax2.plot(hour_rate.index, hour_rate.values, 'k--o', markersize=4, linewidth=1.5,
         label='Fraud Rate')
ax2.set_ylabel('Fraud Rate')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# --- Day of week ---
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_rate  = df.groupby('day_of_week')['is_fraud'].mean().reindex(day_order)
day_rate.plot(kind='bar', ax=axes[0, 1], color=FRAUD_COLOR, edgecolor='none')
axes[0, 1].set_title('Fraud Rate by Day of Week')
axes[0, 1].set_ylabel('Fraud Rate')
axes[0, 1].set_xticklabels([d[:3] for d in day_order], rotation=0)
axes[0, 1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# --- Heatmap: day × hour ---
pivot = df.groupby(['day_of_week', 'hour'])['is_fraud'].mean().unstack()
pivot = pivot.reindex(day_order)
sns.heatmap(pivot, ax=axes[1, 0], cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Fraud Rate'})
axes[1, 0].set_title('Fraud Rate Heatmap — Day × Hour')
axes[1, 0].set_xlabel('Hour')
axes[1, 0].set_ylabel('')

# --- Monthly trend ---
monthly = df.groupby('month_year')['is_fraud'].agg(['mean', 'count']).reset_index()
monthly = monthly.sort_values('month_year')
axes[1, 1].plot(monthly['month_year'], monthly['mean'], 'o-',
                color=FRAUD_COLOR, linewidth=2, markersize=5)
axes[1, 1].set_title('Fraud Rate by Month')
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Fraud Rate')
axes[1, 1].set_xticklabels(monthly['month_year'], rotation=45, ha='right')
axes[1, 1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.suptitle('Temporal Fraud Patterns', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Finding:** Fraud peaks during **late-night / early-morning hours (11 PM – 3 AM)** when monitoring is lower. Weekends show a slightly elevated fraud rate. The day×hour heatmap reveals that **Saturday night** is the highest-risk window.

## 6 — Geographic Analysis <a id='6-geo'></a>

In [ ]:
from src.data.features import add_geo_features
df = add_geo_features(df)

print('Geo-distance statistics (km) by class:')
df.groupby('label')['geo_distance_km'].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution
for lbl, color in PALETTE.items():
    subset = df[df['label'] == lbl]['geo_distance_km'].clip(upper=500)
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=lbl, density=True)
axes[0].set_title('Cardholder–Merchant Distance Distribution (clipped 500 km)')
axes[0].set_xlabel('Distance (km)')
axes[0].set_ylabel('Density')
axes[0].legend()

# CDF
for lbl, color in PALETTE.items():
    vals = np.sort(df[df['label'] == lbl]['geo_distance_km'].clip(upper=1000).values)
    cdf  = np.arange(1, len(vals) + 1) / len(vals)
    axes[1].plot(vals, cdf, color=color, label=lbl, linewidth=2)
axes[1].set_title('CDF of Geo-Distance')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('Cumulative Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fraud sample scatter on US map
fraud_sample = df[df['is_fraud'] == 1].sample(min(3000, df['is_fraud'].sum()), random_state=42)
fig = px.scatter_geo(
    fraud_sample,
    lat='lat', lon='long',
    color_discrete_sequence=[FRAUD_COLOR],
    scope='usa',
    title='Geographic Distribution of Fraudulent Transactions',
    opacity=0.5,
    hover_data={'amt': True, 'category': True},
)
fig.update_layout(height=450)
fig.show()

> **Finding:** Fraudulent transactions occur at **significantly larger cardholder–merchant distances** than legitimate ones. The engineered `geo_distance_km` feature will be one of the most predictive signals in the model.

## 7 — Merchant & Category Analysis <a id='7-merchant'></a>

In [ ]:
cat_stats = df.groupby('category')['is_fraud'].agg(['mean', 'sum', 'count'])
cat_stats.columns = ['fraud_rate', 'fraud_count', 'total']
cat_stats = cat_stats.sort_values('fraud_rate', ascending=False)
print(cat_stats.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fraud rate by category
cat_stats_sorted = cat_stats.sort_values('fraud_rate')
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(cat_stats_sorted)))
axes[0].barh(cat_stats_sorted.index, cat_stats_sorted['fraud_rate'], color=colors)
axes[0].set_title('Fraud Rate by Merchant Category')
axes[0].set_xlabel('Fraud Rate')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Fraud volume by category
cat_vol = cat_stats.sort_values('fraud_count')
axes[1].barh(cat_vol.index, cat_vol['fraud_count'], color=FRAUD_COLOR, alpha=0.75)
axes[1].set_title('Fraud Volume by Merchant Category')
axes[1].set_xlabel('Fraud Transaction Count')

plt.tight_layout()
plt.show()

In [ ]:
# Top merchants by fraud count
top_merch = (
    df[df['is_fraud'] == 1]
    .groupby('merchant')
    .size()
    .nlargest(20)
    .reset_index(name='fraud_count')
)

fig = px.bar(
    top_merch.sort_values('fraud_count'),
    x='fraud_count', y='merchant', orientation='h',
    title='Top 20 Merchants by Fraud Volume',
    color='fraud_count', color_continuous_scale='Reds',
)
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()

> **Finding:** Categories like `shopping_net`, `misc_net`, and `grocery_pos` show the highest fraud **volumes** (high transaction frequency), while `travel` and `entertainment` show elevated **rates**. Target-encoding these categories will add strong signal.

## 8 — Cardholder Demographics <a id='8-demographics'></a>

In [ ]:
df['age'] = ((df['trans_date_trans_time'] - pd.to_datetime(df['dob'])).dt.days / 365.25)

print('Age statistics by class:')
df.groupby('label')['age'].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Age distribution
for lbl, color in PALETTE.items():
    axes[0].hist(df[df['label'] == lbl]['age'].clip(18, 90),
                 bins=40, alpha=0.6, color=color, label=lbl, density=True)
axes[0].set_title('Age Distribution by Class')
axes[0].set_xlabel('Age (years)')
axes[0].legend()

# Fraud rate by gender
gender_rate = df.groupby('gender')['is_fraud'].mean()
axes[1].bar(gender_rate.index, gender_rate.values,
            color=[LEGIT_COLOR, FRAUD_COLOR], width=0.4, edgecolor='white')
axes[1].set_title('Fraud Rate by Gender')
axes[1].set_ylabel('Fraud Rate')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# City population vs fraud
df['log_city_pop'] = np.log1p(df['city_pop'])
for lbl, color in PALETTE.items():
    axes[2].hist(df[df['label'] == lbl]['log_city_pop'],
                 bins=40, alpha=0.6, color=color, label=lbl, density=True)
axes[2].set_title('log(City Population) by Class')
axes[2].set_xlabel('log(city_pop)')
axes[2].legend()

plt.tight_layout()
plt.show()

> **Finding:** Fraud is slightly more prevalent among **older cardholders** (less digital-fraud awareness). Gender shows minimal difference. Smaller cities have marginally higher fraud rates — possibly due to fewer card-present options pushing cardholders online.

## 9 — Feature Correlations <a id='9-correlations'></a>

In [ ]:
num_cols = ['amt', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long',
            'geo_distance_km', 'hour', 'age', 'is_fraud']
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Pearson Correlation Matrix (numeric features)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Point-biserial correlation of each feature with is_fraud
correlations = df[num_cols].corrwith(df['is_fraud']).drop('is_fraud').sort_values(key=abs, ascending=False)
print('Feature correlations with is_fraud (|r| sorted):')
print(correlations.round(4))

> **Finding:** `geo_distance_km` and `amt` show the highest linear correlations with `is_fraud`. Low overall correlations (all < 0.1) confirm that **non-linear models** (tree ensembles, neural networks) are necessary to capture fraud patterns.

## 10 — Engineered Feature Preview <a id='10-features'></a>

In [ ]:
# Sample 5k rows for speed (velocity features are O(n²) per card without vectorisation)
from src.data.features import add_velocity_features, add_spend_features, add_time_features

sample = df.sample(5000, random_state=42).copy()
sample = add_time_features(sample)
sample = add_velocity_features(sample, windows=[7, 30])
sample = add_spend_features(sample)

eng_cols = ['tx_velocity_7d', 'tx_velocity_30d', 'spend_rolling_mean',
            'spend_rolling_std', 'spend_ratio']
print(sample.groupby('label')[eng_cols].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

for ax, col in zip(axes, ['tx_velocity_7d', 'spend_rolling_mean', 'spend_ratio']):
    for lbl, color in PALETTE.items():
        vals = sample[sample['label'] == lbl][col].clip(
            upper=sample[col].quantile(0.99)
        )
        ax.hist(vals, bins=40, alpha=0.6, color=color, label=lbl, density=True)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle('Engineered Feature Distributions — Fraud vs Legit',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Finding:** Fraudulent transactions show **lower velocity** (fraudsters avoid repeated charges on the same card) and a notably **higher `spend_ratio`** (the fraudulent amount is often much larger than that card's historical average spend). These engineered features will substantially improve model performance.

## 11 — Key Findings & Modelling Implications <a id='11-findings'></a>

| # | Finding | Modelling Implication |
|---|---|---|
| 1 | **427:1 class imbalance** | Use SMOTE + `class_weight='balanced'` + AUC-PR as primary metric |
| 2 | **Late-night fraud peak** | `hour` and `is_night` are strong temporal features |
| 3 | **Large geo-distance for fraud** | `geo_distance_km` (Haversine) is the single strongest engineered feature |
| 4 | **High `spend_ratio` for fraud** | Rolling spend normalisation reveals abnormal single transactions |
| 5 | **Low transaction velocity for fraud** | `tx_velocity_7d/30d` helps distinguish unusual one-off charges |
| 6 | **Category-specific fraud rates** | Target-encode `category`, `merchant`, `job` (smoothed mean encoding) |
| 7 | **Low linear correlations** | Non-linear models (XGBoost, FNN, LSTM) required |
| 8 | **Sequential behaviour per card** | LSTM over per-card transaction sequences can catch gradual account takeover |

### Modelling plan
1. **Baseline** — Logistic Regression (interpretable, calibrated probabilities)
2. **Tree ensemble** — Random Forest + XGBoost with Optuna tuning (non-linear, handles mixed types)
3. **Deep learning** — FNN (tabular), LSTM (sequential per-card behaviour)
4. **Ensemble** — Soft-voting / stacking to combine all models' strengths
5. **Explainability** — SHAP values for every model to justify flagged transactions